In [0]:
%python
from pyspark.sql.functions import current_timestamp, col

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation","/Volumes/associate_assignment/default/raw/customer_volume/customer_CDC/schema/")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load("/Volumes/associate_assignment/default/raw/customer_volume/customer_CDC/landing/")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
)

(
    df.writeStream
        .format("delta")
        .option("checkpointLocation", "/Volumes/associate_assignment/default/raw/customer_volume/customer_CDC/checkpoint/")
        .trigger(availableNow=True)
        .toTable("associate_assignment.default.bronze_customer_cdc")
)

In [0]:
select * from associate_assignment.default.bronze_customer_cdc